# Google Earth Engine 数据下载与处理流程

**基于 RUSLE 模型的广东省土壤侵蚀遥感估算**

本 Notebook 演示如何通过 Google Earth Engine 下载并处理遥感数据，用于 RUSLE 模型各因子的计算。

## 环境配置

In [ ]:
# !pip install earthengine-api geemap rasterio geopandas

import ee
import os

# ============================================
# 配置你的 GEE 项目
# ============================================
# 替换为你的 GEE 项目 ID
GEE_PROJECT = 'YOUR_PROJECT_ID'

# 研究区资产（替换为你的资产名称）
ROI_ASSET = 'projects/YOUR_PROJECT_ID/assets/YOUR_ASSET_NAME'

# 输出目录
OUT_DIR = './data'
os.makedirs(OUT_DIR, exist_ok=True)

# 初始化 GEE
try:
    ee.Initialize(project=GEE_PROJECT)
    print('GEE 初始化成功')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('GEE 认证并初始化成功')

## 1. 降水数据下载 (ERA5-Land)

In [ ]:
def download_monthly_precipitation(year_start=1985, year_end=2024):
    """
    下载 ERA5-Land 月总降水量
    
    参数:
        year_start: 起始年份
        year_end: 结束年份
    """
    dataset = (ee.ImageCollection('ECMWF/ERA5_LAND/MONTHLY_AGGR')
                .filterDate(f'{year_start}-01-01', f'{year_end}-12-31')
                .select('total_precipitation_sum'))
    
    roi = ee.FeatureCollection(ROI_ASSET)
    scale = 1000  # ERA5 分辨率
    
    for year in range(year_start, year_end + 1):
        for month in range(1, 13):
            start = ee.Date.fromYMD(year, month, 1)
            end = start.advance(1, 'month')
            
            img = (dataset.filterDate(start, end)
                   .mean()
                   .multiply(1000)  # m -> mm
                   .rename('precipitation_mm')
                   .clip(roi))
            
            out_name = f'ERA5_Precipitation_Monthly_{year}{month:02d}.tif'
            out_path = os.path.join(OUT_DIR, 'precipitation', out_name)
            
            if os.path.exists(out_path):
                print(f'跳过已存在: {out_name}')
                continue
            
            url = img.getDownloadURL({
                'region': roi.geometry(),
                'scale': scale,
                'crs': 'EPSG:4326',
                'format': 'GEO_TIFF'
            })
            # 使用 requests 下载（见 pre.py 脚本）
            print(f'生成下载链接: {out_name}')
    
    return '下载任务已提交，请运行 scripts/01_data_download/pre.py 进行下载'

## 2. 植被指数数据 (MODIS EVI/NDVI)

In [ ]:
def download_annual_vegetation(year_start=2000, year_end=2020):
    """
    下载 MODIS MOD13A2 年度合成 EVI/NDVI
    
    数据源: MODIS/061/MOD13A2
    """
    roi = ee.FeatureCollection(ROI_ASSET).geometry()
    
    def maskQA(img):
        qa = img.select('SummaryQA')
        return img.updateMask(qa.eq(0))  # 仅保留高质量像元
    
    for year in range(year_start, year_end + 1):
        col = (ee.ImageCollection('MODIS/061/MOD13A2')
                .filterDate(f'{year}-01-01', f'{year}-12-31')
                .filterBounds(roi)
                .map(maskQA)
                .select(['NDVI', 'EVI']))
        
        composite = col.mean().multiply(0.0001)  # 缩放因子
        
        for band in ['NDVI', 'EVI']:
            img = composite.select(band)
            out_name = f'{band}_{year}.tif'
            out_path = os.path.join(OUT_DIR, 'vegetation', out_name)
            
            if os.path.exists(out_path):
                print(f'跳过: {out_name}')
                continue
            
            url = img.getDownloadURL({
                'scale': 500,
                'crs': 'EPSG:4326',
                'region': roi.getInfo(),
                'fileFormat': 'GeoTIFF'
            })
            print(f'生成: {out_name}')
    
    return '请运行 scripts/01_data_download/main.py 进行下载'

## 3. RUSLE 各因子计算

In [ ]:
# 参考 scripts/02_data_processing/ 中的脚本:
# - RUSLE_R.py  : R 因子（降雨侵蚀力）
# - RUSLE_K.py  : K 因子（土壤可蚀性）
# - RUSLE_LS.py : LS 因子（坡长坡度）
# - RUSLE_C.py  : C 因子（植被覆盖）
# - RUSLE_P.py  : P 因子（水土保持措施）

## 4. RUSLE 主方程

In [ ]:
# from scripts.RUSLE import RUSLECalculator
# 
# calc = RUSLECalculator(target_resolution=30, target_crs='EPSG:4547')
# calc.load_gd_boundary('./data/GD_boundary.shp')
# calc.compute_all_factors()
# erosion = calc.compute_RUSLE()